In [5]:
import base64
from io import BytesIO
from PIL import Image

def encode_image(pil_img):
    buffer = BytesIO()
    pil_img.save(buffer, format="JPEG")

    return base64.b64encode(buffer.getvalue()).decode("utf-8")

In [2]:
from transformers import Qwen3VLMoeForConditionalGeneration, AutoProcessor

model = Qwen3VLMoeForConditionalGeneration.from_pretrained(
    "Qwen/Qwen3-VL-30B-A3B-Instruct", dtype="auto", device_map="auto"
)

processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-30B-A3B-Instruct")

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/882 [00:00<?, ?it/s]

In [ ]:

concept = "Person"

image = Image.open("/workspace/Dataset-Curation/curated_data/image/clean_dataset/person/000000009448.jpg")

base64_image = encode_image(image)

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": f"data:image/jpeg;base64,{base64_image}",
            },
            {"type": "text", 
             "text": f"Does the green box include {concept}? Answer only with 1 for yes or 0 for no, no other text."},
        ],
    }
]

# Preparation for inference
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)

inputs = inputs.to(model.device)



import torch
with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=128)

generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)

print(output_text[0])


1
